# DeepSeek-V4.1-Flash-1017 Uncensored GSQ-RCO Coder (G4 / H100 / A100 · Auto Max Context up to 1M + Vision)

Serves `Abiray/DeepSeek-V4.1-Flash-1017-Uncensored-GSQ-RCO-Coder-GGUF` (50% expert-pruned 192-expert `IQ2_XXS` MoE trunk, 72.1 GB + 110.6 GB `IQ4_NL` Engram lookup table + BF16 `deepseek41v` vision projector) via `llama-server --api-key-file` on a Colab G4 GPU by default (100% GPU offload + full 1,048,576 native 1M context + 100% DDR5 RAM-resident Engram + default `--reasoning on` with per-request client thinking control), and automatic SSD lazy fallback on lower-RAM GPUs.

Run it with the service script (`colabllm <name> up`), which creates the Colab session, mounts Drive with `colab dfs`, optionally creates a named Cloudflare tunnel, and injects `CT_*` settings. Cell order is the standard contract -- see `.agents/skills/colab-llm-service/SKILL.md`.

In [ ]:
# @title 0. Config -- values come from `colab exec --env` (defaults allow running in the Colab UI)
import json, os, re, secrets, shutil, stat, subprocess, sys, time, urllib.request, urllib.error

SERVICE      = os.environ.get("CT_SERVICE", "deepseek-coder")
SERVICE_PORT = int(os.environ.get("CT_SERVICE_PORT", "8001"))
AUTH         = os.environ.get("CT_AUTH", "builtin")            # caddy | builtin | none
PUBLIC_PORT  = int(os.environ.get("CT_PUBLIC_PORT", "8001"))  # what cloudflared targets
AUTH_USER    = os.environ.get("CT_AUTH_USER", "admin")
AUTH_PASS    = os.environ.get("CT_AUTH_PASS") or secrets.token_urlsafe(18)
API_TOKEN    = os.environ.get("CT_API_TOKEN") or "sk-" + secrets.token_urlsafe(32)
TUNNEL_TOKEN = os.environ.get("CT_TUNNEL_TOKEN", "")            # set -> fixed domain (named tunnel)
PUBLIC_HOST  = os.environ.get("CT_PUBLIC_HOSTNAME", "")
CREDS_FROM_ENV = bool(os.environ.get("CT_AUTH_PASS"))

WORK = "/content/ct"                                            # logs + binaries of this notebook
os.makedirs(WORK, exist_ok=True)
for _p in ("/content/drive/MyDrive/colab-llm/hf_token.txt",
           "/content/drive/MyDrive/colab-services/hf_token.txt",
           "/content/drive/MyDrive/colab-services/qwen-flash/hf_token.txt"):
    if not os.environ.get("HF_TOKEN") and os.path.isfile(_p):
        _tok = open(_p).read().strip()
        if _tok:
            os.environ["HF_TOKEN"] = os.environ["HUGGING_FACE_HUB_TOKEN"] = _tok
            break
MOUNT_DRIVE = os.environ.get("CT_MOUNT_DRIVE", "1") != "0"
DATA_DIR = os.environ.get("CT_DATA_DIR", f"/content/drive/MyDrive/colab-llm/{SERVICE}")
if not MOUNT_DRIVE or not os.path.isdir("/content/drive/MyDrive"):   # Drive not mounted -> local fallback
    if DATA_DIR.startswith("/content/drive/"):
        DATA_DIR = f"/content/data/{SERVICE}"
os.makedirs(DATA_DIR, exist_ok=True)
PROCS = {}                                                      # name -> (Popen, restart function)

def ct_spawn(name, args, env=None, cwd=None):
    """Start a background process; output goes to /content/ct/<name>.log."""
    with open(f"{WORK}/{name}.log", "a") as logf:   # child keeps its own dup of the fd
        return subprocess.Popen(args, stdout=logf, stderr=subprocess.STDOUT, cwd=cwd,
                                env={**os.environ, **(env or {})}, start_new_session=True)

def ct_stop_proc(proc, timeout=15):
    """Terminate a ct_spawn process group, escalating to SIGKILL after a bounded wait."""
    import signal
    if proc is None or proc.poll() is not None:
        return
    for sig in (signal.SIGTERM, signal.SIGKILL):
        try:
            os.killpg(proc.pid, sig)
        except (ProcessLookupError, PermissionError):
            try:
                proc.send_signal(sig)
            except ProcessLookupError:
                return
        try:
            proc.wait(timeout=timeout)
            return
        except subprocess.TimeoutExpired:
            continue

def ct_gpu_stat(query="memory.used,memory.total,utilization.gpu"):
    """nvidia-smi summary that never raises (CPU runtimes / transient driver errors)."""
    try:
        return subprocess.check_output(["nvidia-smi", f"--query-gpu={query}", "--format=csv,noheader"],
                                       text=True, stderr=subprocess.DEVNULL, timeout=20).strip()
    except (OSError, subprocess.SubprocessError):
        return "cpu/no-gpu"

def ct_log_size(name):
    p = f"{WORK}/{name}.log"
    return os.path.getsize(p) if os.path.exists(p) else 0

def ct_log_since(name, offset):
    p = f"{WORK}/{name}.log"
    if not os.path.exists(p):
        return ""
    with open(p, "rb") as handle:
        handle.seek(min(offset, os.path.getsize(p)))
        return handle.read().decode("utf-8", errors="replace")

def ct_public_gate(public_url, probe, verify_auth, log, timeout=120, interval=8):
    """Wait for the public endpoint; auth failures are fatal, network errors are retried."""
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        try:
            code = probe()
        except Exception as err:   # DNS / edge propagation / TLS warm-up
            log(f"Public probe pending ({type(err).__name__}: {err})")
            time.sleep(interval)
            continue
        if code == 200:
            verify_auth()            # RuntimeError propagates: never report CT_READY with broken auth
            return
        log(f"Public probe pending (HTTP {code})")
        time.sleep(interval)
    raise RuntimeError(f"Public endpoint {public_url} failed readiness verification within {timeout}s")

def ct_watchdog(health, start_service, log, *, service_key="service", log_name="service",
                interval=60, grace=900, dead_markers=(), heartbeat=None):
    """Supervise PROCS forever: restart dead processes, debounce health failures after reloads."""
    started = time.monotonic()
    failures = 0
    grace_until = time.monotonic()
    log_offset = ct_log_size(log_name)

    def restart_service(reason):
        nonlocal failures, grace_until, log_offset
        log(f"[watchdog] {reason}; restarting {service_key}")
        ct_stop_proc(PROCS[service_key][0])
        PROCS[service_key] = (start_service(), start_service)
        failures = 0
        grace_until = time.monotonic() + grace
        log_offset = ct_log_size(log_name)

    while True:
        time.sleep(interval)
        for name, (proc, restart) in list(PROCS.items()):
            if proc.poll() is not None:
                log(f"[watchdog] {name} exited with {proc.returncode}; restarting\n{ct_tail(name, 20)}")
                PROCS[name] = (restart(), restart)
                if name == service_key:
                    failures = 0
                    grace_until = time.monotonic() + grace
                    log_offset = ct_log_size(log_name)
        if dead_markers:
            fresh = ct_log_since(log_name, log_offset)
            log_offset += len(fresh.encode("utf-8", errors="replace"))
            if any(marker in fresh for marker in dead_markers):
                restart_service("Detected dead worker marker in service log")
                continue
        try:
            healthy = bool(health())
        except Exception:
            healthy = False
        if healthy:
            failures = 0
        elif time.monotonic() >= grace_until:
            failures += 1
        if failures >= 6:
            restart_service("6 consecutive failed health checks")
        if heartbeat is not None:
            try:
                heartbeat(int(time.monotonic() - started))
            except Exception as err:
                log(f"[watchdog] heartbeat warning: {err}")

def ct_wait_http(url, timeout=600, ok=lambda code: code < 500):
    """Wait until url answers with a non-5xx status (401/404 count as 'listening')."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if ok(r.status): return r.status
        except urllib.error.HTTPError as e:
            if ok(e.code): return e.code
        except Exception:
            pass
        time.sleep(2)
    raise TimeoutError(f"{url} not ready after {timeout}s")

def ct_tail(name, n=30):
    p = f"{WORK}/{name}.log"
    return "".join(open(p, errors="replace").readlines()[-n:]) if os.path.exists(p) else ""

def ct_fetch_binary(path, url):
    if not os.path.exists(path):
        partial = f"{path}.{os.getpid()}.partial"
        with urllib.request.urlopen(url, timeout=60) as src, open(partial, "wb") as dst:
            shutil.copyfileobj(src, dst)
        os.chmod(partial, os.stat(partial).st_mode | stat.S_IEXEC)
        os.replace(partial, path)
    return path

print(f"service={SERVICE} port={SERVICE_PORT} auth={AUTH} public_port={PUBLIC_PORT} "
      f"tunnel={'named ' + PUBLIC_HOST if TUNNEL_TOKEN else 'quick'} data={DATA_DIR}")


In [ ]:
# @title 1. Setup -- parallel llama.cpp CUDA build/cache + 183.7 GB Coder GGUF + Engram shard-6 symlink + hardware-max context plan
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

DS_WORK = Path(os.environ.get("CT_DS_WORK_DIR", "/content/deepseek-coder"))
DS_WORK.mkdir(parents=True, exist_ok=True)
(DS_WORK / "deepseek_coder_runtime.py").write_text('"""Hardware-adaptive planning, model staging, and llama.cpp runtime helpers for DeepSeek-V4.1-Flash Coder.\n\nTargets `Abiray/DeepSeek-V4.1-Flash-1017-Uncensored-GSQ-RCO-Coder-GGUF` (50% expert-pruned 192-expert\n`IQ2_XXS` MoE trunk + `IQ4_NL` Engram conditional memory lookup table + `BF16` `deepseek41v` vision\nprojector) served via `llama-server` with native `--api-key-file` Bearer authentication.\n\nImported both locally (CLI `plan` / validation / unit tests, standard library only) and inside the\ngenerated Colab notebook (`src/colabllm/services/deepseek-coder.ipynb`).\n"""\nfrom __future__ import annotations\n\nfrom concurrent.futures import ThreadPoolExecutor\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport platform\nimport re\nimport shutil\nimport struct\nimport subprocess\nimport sys\nimport tarfile\nimport time\nfrom typing import Any\n\nDEFAULT_REPO = "Abiray/DeepSeek-V4.1-Flash-1017-Uncensored-GSQ-RCO-Coder-GGUF"\nDEFAULT_REVISION = "cfe5da6df1b4e22baf864e33f5132a6202c821e3"\nDEFAULT_QUANT = "IQ2_XXS"\nDEFAULT_ALIAS = "deepseek-v4.1-flash-coder"\nDEFAULT_LLAMA_REPO = "https://github.com/ggml-org/llama.cpp.git"\nDEFAULT_LLAMA_COMMIT = "3259ab6e5dd86fad1931962d784702cb53c7877b"\n\nMODEL_MAX_CONTEXT = 1_048_576\nMIN_CONTEXT = 16_384\nCONTEXT_ALIGN = 16_384\nMODEL_NUM_LAYERS = 40\n\n# Exact weight footprints of the 192-expert IQ2_XXS Coder build (MiB = 1024**2 bytes):\n#   - 5 trunk shards (00001..00005): 72,123,567,456 B = 68,782.4 MiB\n#       * Dense layers (tok/out embd, MLA/DSA attention, HC, shared experts, engram gates): ~15,000 MiB\n#       * Routed MoE experts (40 layers x 192 experts): ~53,783 MiB (~1,345 MiB / layer)\n#   - BF16 deepseek41v mmproj: 972,208,096 B = 927.2 MiB\n#   - IQ4_NL Engram table (shard 6): 110,595,290,816 B = 105,471.9 MiB\n#       * On high-RAM hosts (>=128 GiB free after CPU MoE, e.g. G4 176 GiB / H100 167 GiB), loaded 100%\n#         resident into DDR5 host RAM (`--load-mode none --lazy-mode off`) to eliminate cloud-disk page faults.\n#       * On smaller-RAM hosts (e.g. A100-40G 83.5 GiB), streamed via `--load-mode mmap --lazy-mode on`.\nTRUNK_TOTAL_MIB = 68_783\nDENSE_WEIGHTS_MIB = 15_000\nMOE_LAYER_MIB = 1_345\nVISION_MMPROJ_MIB = 928\nENGRAM_FILE_MIB = 105_472\nENGRAM_RAM_HEADROOM_MIB = 25_600\nFIXED_GPU_RUNTIME_RESERVE_MIB = 5_632\nSYSTEM_RAM_RESERVE_MIB = 10_240\nFULL_GPU_MIN_VRAM_MIB = 76_800\nREASONING_EFFORT_LEVELS: tuple[str, ...] = ("minimal", "low", "medium", "high", "xhigh", "max")\n\nENGRAM_FILENAME = "DeepSeek-V4.1-Flash-Engram-IQ4_NL.gguf"\nSHARD6_LINK_NAME = "DeepSeek-V4.1-Flash-1017-Uncensored-GSQ-RCO-Coder-IQ2_XXS-00006-of-00006.gguf"\nVISION_PROJECTOR_SPEC: dict[str, Any] = {\n    "name": "mmproj-DeepSeek-V4.1-Flash-1017-BF16.gguf",\n    "revision": DEFAULT_REVISION,\n    "size": 972_208_096,\n    "sha256": "0fdc4b7693f287e9665940cf42bd39c546ffe4f0def89c569d7d0e45d60d2df1",\n}\n\nDEFAULT_MODEL_LOCK: dict[str, Any] = {\n    "repo": DEFAULT_REPO,\n    "revision": DEFAULT_REVISION,\n    "shard6_link": SHARD6_LINK_NAME,\n    "engram_file": ENGRAM_FILENAME,\n    "files": [\n        {\n            "name": "DeepSeek-V4.1-Flash-1017-Uncensored-GSQ-RCO-Coder-IQ2_XXS-00001-of-00006.gguf",\n            "size": 5_780_704,\n            "sha256": "108e63d27240da2d99846250727a7b282e05a7fdf2448435e81904b9a98d8a4a",\n        },\n        {\n            "name": "DeepSeek-V4.1-Flash-1017-Uncensored-GSQ-RCO-Coder-IQ2_XXS-00002-of-00006.gguf",\n            "size": 19_776_064_960,\n            "sha256": "a503855d07b92bb9cf87583791788019cc5dcf57e46c9f80b99ae597b69fc880",\n        },\n        {\n            "name": "DeepSeek-V4.1-Flash-1017-Uncensored-GSQ-RCO-Coder-IQ2_XXS-00003-of-00006.gguf",\n            "size": 19_939_989_696,\n            "sha256": "a402c6d3ff9ebe5213cef3e59c6e7bb95f00f749d07f5491276acc35d01bf0db",\n        },\n        {\n            "name": "DeepSeek-V4.1-Flash-1017-Uncensored-GSQ-RCO-Coder-IQ2_XXS-00004-of-00006.gguf",\n            "size": 19_682_533_472,\n            "sha256": "ca5e46dada8e6dd915532ddb0b5dd779d7cc59402ef8cb08fc0030671c559f43",\n        },\n        {\n            "name": "DeepSeek-V4.1-Flash-1017-Uncensored-GSQ-RCO-Coder-IQ2_XXS-00005-of-00006.gguf",\n            "size": 12_719_198_624,\n            "sha256": "f375be5b2b08d0f4cacbc09462b97f98e674c5a91e7104815d3b4b9139204f74",\n        },\n        {\n            "name": ENGRAM_FILENAME,\n            "size": 110_595_290_816,\n            "sha256": "3dac4c54893ad7aa89103c0028500eadc78e74112cef3a2930a97e67dc9fb4f3",\n        },\n    ],\n}\n\nCACHE_TYPE_BYTES: dict[str, float] = {\n    "f32": 4.0,\n    "f16": 2.0,\n    "bf16": 2.0,\n    "q8_0": 1.0625,\n    "q5_1": 0.75,\n    "q5_0": 0.6875,\n    "q4_1": 0.625,\n    "q4_0": 0.5625,\n    "iq4_nl": 0.5625,\n}\n\nSTANDARD_CONTEXT_RUNGS: tuple[int, ...] = (\n    1_048_576,\n    917_504,\n    786_432,\n    655_360,\n    524_288,\n    393_216,\n    262_144,\n    196_608,\n    131_072,\n    65_536,\n    32_768,\n    16_384,\n)\n\nOOM_LOG_MARKERS: tuple[str, ...] = (\n    "out of memory",\n    "cuda_malloc failed",\n    "failed to allocate",\n    "ggml_backend_cuda_buffer_type_alloc_buffer",\n    "unable to allocate",\n    "not enough memory",\n)\n\n_SHARD_FIRST_RE = re.compile(r"^(.*)-0*1-of-(0*\\d+)\\.gguf$", re.IGNORECASE)\n\n\ndef dsv41_kv_bytes_per_token(cache_type: str = "f16", parallel: int = 1) -> float:\n    """Exact persistent DSV4.1 compressed KV + lightning-indexer K cache bytes per token.\n\n    In `llama_kv_cache_dsv4` (`LLM_ARCH_DEEPSEEK41`):\n      - `kv_raw` uses sliding-window attention (`n_swa = 128`) across all 40 layers (~40 MiB fixed).\n      - Only `kv_source_layers = [2, 8, 14, 20]` own context-proportional `[kv(512) | index_k(128)]` rows:\n          * 3 CSA source layers (`ratio = 2`): `3 * 0.5 * 640 = 960` elements/token\n          * 1 HCA source layer (`ratio = 1`):  `1 * 1.0 * 640 = 640` elements/token\n        Total = `1,600` elements/token (`3,200 B/tok` at `f16`, i.e. `3,200 MiB` at 1M context).\n    """\n    elem_bytes = CACHE_TYPE_BYTES.get(cache_type.strip().lower(), 2.0)\n    slots = max(1, int(parallel))\n    return 1600.0 * elem_bytes * slots\n\n\ndef dsv41_total_vram_bytes_per_token(\n    ubatch: int = 128,\n    cache_type: str = "f16",\n    parallel: int = 1,\n) -> float:\n    """Conservative per-token GPU budget including persistent KV cache + `sched_reserve` graph workspace.\n\n    During worst-case reservation (`nt = n_ubatch > 32`), `deepseek41::build_attention41` allocates\n    contiguous `comp_k` / `k_all` views plus `[n_comp, ubatch]` masks and candidate block scores\n    (source analysis ~14 B/tok * ubatch; budgeted conservatively at 28 B/tok * ubatch + 15% safety margin).\n    """\n    kv_bpt = dsv41_kv_bytes_per_token(cache_type=cache_type, parallel=parallel)\n    ub = max(32, int(ubatch))\n    # quantized KV caches dequantize `ik` and `comp_k` to F32 during attention, adding temporary workspace\n    is_quant_kv = cache_type.strip().lower() not in ("f16", "bf16", "f32")\n    comp_k_bpt = 4096.0 if is_quant_kv else 2048.0\n    mask_and_score_bpt = 28.0 * ub\n    return (kv_bpt + comp_k_bpt + mask_and_score_bpt) * 1.15\n\n\ndef align_context_tokens(tokens: int, *, minimum: int = MIN_CONTEXT, maximum: int = MODEL_MAX_CONTEXT) -> int:\n    """Clamp and floor-align token count to a clean 16K boundary (`1,048,576` max)."""\n    if tokens >= maximum:\n        return maximum\n    if tokens <= minimum:\n        return minimum\n    aligned = (int(tokens) // CONTEXT_ALIGN) * CONTEXT_ALIGN\n    return max(minimum, min(maximum, aligned))\n\n\ndef parse_context_spec(raw: str | int | None) -> int | None:\n    """Parse `auto` -> `None`, or a positive integer token count (`16384..1048576`, supports `256k`/`1m`)."""\n    if raw is None:\n        return None\n    text = str(raw).strip().lower().replace("_", "")\n    if not text or text == "auto":\n        return None\n    mult = 1\n    if text.endswith("k"):\n        mult = 1024\n        text = text[:-1]\n    elif text.endswith("m"):\n        mult = 1024 * 1024\n        text = text[:-1]\n    try:\n        value = int(text) * mult\n    except ValueError as err:\n        raise ValueError(f"invalid --context value {raw!r}; expected \'auto\' or integer in {MIN_CONTEXT}..{MODEL_MAX_CONTEXT}") from err\n    if value < MIN_CONTEXT or value > MODEL_MAX_CONTEXT:\n        raise ValueError(f"--context must be between {MIN_CONTEXT} and {MODEL_MAX_CONTEXT} (got {value})")\n    return value\n\n\ndef parse_cpu_moe_spec(raw: str | int | None) -> int | None:\n    """Parse `auto` -> `None`, or `0..40` MoE layers to offload to CPU RAM (`--n-cpu-moe`)."""\n    if raw is None:\n        return None\n    text = str(raw).strip().lower()\n    if not text or text == "auto":\n        return None\n    try:\n        value = int(text)\n    except ValueError as err:\n        raise ValueError(f"invalid --cpu-moe-layers value {raw!r}; expected \'auto\' or integer 0..{MODEL_NUM_LAYERS}") from err\n    if value < 0 or value > MODEL_NUM_LAYERS:\n        raise ValueError(f"--cpu-moe-layers must be between 0 and {MODEL_NUM_LAYERS} (got {value})")\n    return value\n\n\ndef parse_ubatch_spec(raw: str | int | None) -> int | None:\n    """Parse `auto` -> `None`, or a positive integer micro-batch size (`32..4096`)."""\n    if raw is None:\n        return None\n    text = str(raw).strip().lower()\n    if not text or text == "auto":\n        return None\n    try:\n        value = int(text)\n    except ValueError as err:\n        raise ValueError(f"invalid --ubatch value {raw!r}; expected \'auto\' or integer 32..4096") from err\n    if value < 32 or value > 4096:\n        raise ValueError(f"--ubatch must be between 32 and 4096 (got {value})")\n    return value\n\n\ndef build_context_fallback_ladder(planned_context: int, *, allow_stepdown: bool = True) -> list[int]:\n    """Ordered context candidates starting at `planned_context`, stepping down on startup OOM."""\n    planned = max(MIN_CONTEXT, min(MODEL_MAX_CONTEXT, int(planned_context)))\n    if not allow_stepdown:\n        return [planned]\n    rungs = [planned]\n    for rung in STANDARD_CONTEXT_RUNGS:\n        if MIN_CONTEXT <= rung < planned and rung not in rungs:\n            rungs.append(rung)\n    return rungs\n\n\ndef plan_hardware_config(\n    *,\n    vram_mib: int,\n    ram_mib: int,\n    requested_context: str | int | None = "auto",\n    parallel: int = 1,\n    cache_type: str = "f16",\n    enable_vision: bool = True,\n    batch: int = 4096,\n    requested_ubatch: str | int | None = "auto",\n    requested_cpu_moe: str | int | None = "auto",\n) -> dict[str, Any]:\n    """Compute maximum safe context window, GPU/CPU MoE placement, and Engram RAM residency."""\n    vram_mib = max(0, int(vram_mib))\n    ram_mib = max(0, int(ram_mib))\n    parallel = max(1, int(parallel))\n    batch = max(64, int(batch))\n    cache_type = (cache_type or "f16").strip().lower()\n    if cache_type not in CACHE_TYPE_BYTES:\n        raise ValueError(f"unsupported KV cache type {cache_type!r}; choose from {\', \'.join(CACHE_TYPE_BYTES)}")\n\n    explicit_ctx = parse_context_spec(requested_context)\n    explicit_moe = parse_cpu_moe_spec(requested_cpu_moe)\n    explicit_ubatch = parse_ubatch_spec(requested_ubatch)\n\n    vision_mib = VISION_MMPROJ_MIB if enable_vision else 0\n    full_weights_mib = TRUNK_TOTAL_MIB + vision_mib\n    max_ram_for_moe_mib = max(0, ram_mib - SYSTEM_RAM_RESERVE_MIB)\n\n    # CPU-only host (`--gpu none` or no CUDA device)\n    if vram_mib <= 0:\n        min_cpu_ram = full_weights_mib + SYSTEM_RAM_RESERVE_MIB + 2_048\n        if ram_mib < min_cpu_ram:\n            raise ValueError(\n                f"Insufficient system RAM ({ram_mib / 1024:.1f} GiB) for CPU-only loading of the "\n                f"72.1 GB Coder trunk (requires >= {min_cpu_ram / 1024:.1f} GiB RAM; Engram stays on SSD)."\n            )\n        ubatch = explicit_ubatch or min(batch, 128)\n        bpt = dsv41_total_vram_bytes_per_token(ubatch=ubatch, cache_type=cache_type, parallel=parallel)\n        budget_mib = max(0, ram_mib - full_weights_mib - SYSTEM_RAM_RESERVE_MIB)\n        max_tokens = int((budget_mib * 1024 * 1024) // bpt)\n        if explicit_ctx is not None:\n            context = explicit_ctx\n        else:\n            context = align_context_tokens(max_tokens)\n        engram_in_ram = (ram_mib - full_weights_mib) >= (ENGRAM_FILE_MIB + ENGRAM_RAM_HEADROOM_MIB)\n        return {\n            "mode": "cpu-only",\n            "gpu_layers": "0",\n            "n_cpu_moe": 0,\n            "engram_in_ram": engram_in_ram,\n            "context": context,\n            "context_auto": explicit_ctx is None,\n            "context_ladder": build_context_fallback_ladder(context, allow_stepdown=explicit_ctx is None),\n            "batch": max(batch, ubatch),\n            "ubatch": ubatch,\n            "parallel": parallel,\n            "cache_type": cache_type,\n            "enable_vision": enable_vision,\n            "vram_mib": vram_mib,\n            "ram_mib": ram_mib,\n            "estimated_gpu_weights_mib": 0,\n            "estimated_cpu_moe_mib": full_weights_mib,\n            "context_vram_budget_mib": budget_mib,\n            "kv_cache_mib": round(context * dsv41_kv_bytes_per_token(cache_type, parallel) / (1024 * 1024), 1),\n        }\n\n    # GPU host: determine `ubatch` and `n_cpu_moe`\n    if explicit_ubatch is not None:\n        ubatch = min(batch, explicit_ubatch)\n    elif vram_mib >= 90_000:\n        # 96 GB G4 has ~22 GiB free above 100% GPU weights -> 512 ubatch doubles prefill speed (~82.5 GiB total at 1M ctx)\n        ubatch = min(batch, 512)\n    else:\n        # 80 GB H100/A100 or smaller GPUs: 128 ubatch minimizes attention mask workspace to maximize n_ctx\n        ubatch = min(batch, 128)\n\n    bpt = dsv41_total_vram_bytes_per_token(ubatch=ubatch, cache_type=cache_type, parallel=parallel)\n    min_ctx_vram_mib = math.ceil((MIN_CONTEXT * bpt) / (1024 * 1024))\n\n    if explicit_moe is not None:\n        n_cpu_moe = explicit_moe\n    elif vram_mib >= FULL_GPU_MIN_VRAM_MIB:\n        # 80 GB+ GPUs (G4 96 GB, H100/A100 80 GB): keep 100% of weights on GPU by default for maximum tok/s.\n        n_cpu_moe = 0\n        if explicit_ctx is not None:\n            needed_ctx_mib = math.ceil((explicit_ctx * bpt) / (1024 * 1024))\n            shortfall_mib = (full_weights_mib + FIXED_GPU_RUNTIME_RESERVE_MIB + needed_ctx_mib) - vram_mib\n            if shortfall_mib > 0:\n                # Offload just enough MoE layers so an explicitly requested 1M context fits on 80 GB VRAM.\n                n_cpu_moe = min(MODEL_NUM_LAYERS, math.ceil(shortfall_mib / MOE_LAYER_MIB))\n    else:\n        # Sub-75 GiB GPU (e.g. 40 GB A100): keep dense attention + KV cache on GPU and offload MoE layers to RAM.\n        target_ctx_tokens = explicit_ctx if explicit_ctx is not None else 262_144\n        target_ctx_mib = math.ceil((target_ctx_tokens * bpt) / (1024 * 1024))\n        desired_gpu_weights = vram_mib - FIXED_GPU_RUNTIME_RESERVE_MIB - target_ctx_mib\n        min_gpu_weights = DENSE_WEIGHTS_MIB + vision_mib\n        if desired_gpu_weights < min_gpu_weights:\n            # Squeeze target context down to MIN_CONTEXT before giving up\n            desired_gpu_weights = vram_mib - FIXED_GPU_RUNTIME_RESERVE_MIB - min_ctx_vram_mib\n        if desired_gpu_weights < min_gpu_weights:\n            raise ValueError(\n                f"GPU VRAM ({vram_mib / 1024:.1f} GiB) is too small to hold dense attention weights + "\n                f"runtime buffers (~{(min_gpu_weights + FIXED_GPU_RUNTIME_RESERVE_MIB) / 1024:.1f} GiB minimum; "\n                f"use G4 96 GB, H100 80 GB, or A100 40/80 GB)."\n            )\n        shortfall_mib = max(0, full_weights_mib - desired_gpu_weights)\n        n_cpu_moe = min(MODEL_NUM_LAYERS, math.ceil(shortfall_mib / MOE_LAYER_MIB))\n\n    cpu_moe_mib = min(TRUNK_TOTAL_MIB - DENSE_WEIGHTS_MIB, n_cpu_moe * MOE_LAYER_MIB)\n    if cpu_moe_mib > max_ram_for_moe_mib:\n        raise ValueError(\n            f"Hardware requires offloading {n_cpu_moe} MoE layers (~{cpu_moe_mib / 1024:.1f} GiB) to CPU RAM, "\n            f"but only {max_ram_for_moe_mib / 1024:.1f} GiB usable RAM is available "\n            f"(VRAM={vram_mib / 1024:.1f} GiB, RAM={ram_mib / 1024:.1f} GiB). Use `--gpu G4` or `--gpu H100`/`A100`."\n        )\n\n    gpu_weights_mib = full_weights_mib - cpu_moe_mib\n    vram_for_ctx_mib = vram_mib - gpu_weights_mib - FIXED_GPU_RUNTIME_RESERVE_MIB\n    if vram_for_ctx_mib < min_ctx_vram_mib:\n        raise ValueError(\n            f"Insufficient free GPU VRAM ({vram_for_ctx_mib} MiB left after {gpu_weights_mib} MiB GPU weights "\n            f"and {FIXED_GPU_RUNTIME_RESERVE_MIB} MiB reserve); need at least {min_ctx_vram_mib} MiB for "\n            f"{MIN_CONTEXT} context tokens."\n        )\n\n    max_auto_tokens = int((vram_for_ctx_mib * 1024 * 1024) // bpt)\n    if explicit_ctx is not None:\n        needed_mib = math.ceil((explicit_ctx * bpt) / (1024 * 1024))\n        if needed_mib > vram_for_ctx_mib:\n            raise ValueError(\n                f"Requested --context {explicit_ctx} needs ~{needed_mib} MiB VRAM, exceeding available "\n                f"budget ({vram_for_ctx_mib} MiB). Use `--context auto` (selects {align_context_tokens(max_auto_tokens)}) "\n                f"or increase `--cpu-moe-layers`."\n            )\n        context = explicit_ctx\n    else:\n        context = align_context_tokens(max_auto_tokens)\n\n    engram_in_ram = (ram_mib - cpu_moe_mib) >= (ENGRAM_FILE_MIB + ENGRAM_RAM_HEADROOM_MIB)\n    # On startup OOM, step ubatch down (512 -> 256) before sacrificing context window\n    ubatch_ladder = [ubatch]\n    if explicit_ubatch is None and ubatch > 256:\n        ubatch_ladder.append(256)\n    mode = "full-gpu" if n_cpu_moe == 0 else f"hybrid-cpu-moe-{n_cpu_moe}"\n    return {\n        "mode": mode,\n        "gpu_layers": "all",\n        "n_cpu_moe": n_cpu_moe,\n        "engram_in_ram": engram_in_ram,\n        "context": context,\n        "context_auto": explicit_ctx is None,\n        "context_ladder": build_context_fallback_ladder(context, allow_stepdown=explicit_ctx is None),\n        "batch": max(batch, ubatch),\n        "ubatch": ubatch,\n        "ubatch_ladder": ubatch_ladder,\n        "parallel": parallel,\n        "cache_type": cache_type,\n        "enable_vision": enable_vision,\n        "vram_mib": vram_mib,\n        "ram_mib": ram_mib,\n        "estimated_gpu_weights_mib": gpu_weights_mib,\n        "estimated_cpu_moe_mib": cpu_moe_mib,\n        "context_vram_budget_mib": vram_for_ctx_mib,\n        "kv_cache_mib": round(context * dsv41_kv_bytes_per_token(cache_type, parallel) / (1024 * 1024), 1),\n    }\n\n\ndef detect_host_ram_mib() -> int:\n    """Read total system RAM (or cgroup memory.max cap when lower) in MiB on Linux/macOS."""\n    total_bytes = 0\n    meminfo = Path("/proc/meminfo")\n    if meminfo.is_file():\n        for line in meminfo.read_text(encoding="utf-8", errors="replace").splitlines():\n            if line.startswith("MemTotal:"):\n                parts = line.split()\n                if len(parts) >= 2 and parts[1].isdigit():\n                    total_bytes = int(parts[1]) * 1024\n                break\n    if total_bytes <= 0:\n        try:\n            pages = os.sysconf("SC_PHYS_PAGES")\n            page_size = os.sysconf("SC_PAGE_SIZE")\n            if isinstance(pages, int) and isinstance(page_size, int) and pages > 0 and page_size > 0:\n                total_bytes = pages * page_size\n        except (AttributeError, ValueError, OSError):\n            total_bytes = 0\n    for cg_path in (Path("/sys/fs/cgroup/memory.max"), Path("/sys/fs/cgroup/memory/memory.limit_in_bytes")):\n        if cg_path.is_file():\n            raw = cg_path.read_text(encoding="utf-8", errors="replace").strip()\n            if raw.isdigit():\n                cg_val = int(raw)\n                # Ignore unlimited sentinels (e.g. 2^63 - page_size)\n                if 0 < cg_val < (1 << 50):\n                    total_bytes = min(total_bytes, cg_val) if total_bytes > 0 else cg_val\n    return total_bytes // (1024 * 1024)\n\n\ndef detect_gpu_info() -> dict[str, Any]:\n    """Query primary NVIDIA GPU name, compute capability, CUDA arch token, and total VRAM in MiB."""\n    try:\n        out = subprocess.check_output(\n            ["nvidia-smi", "--query-gpu=name,compute_cap,memory.total", "--format=csv,noheader,nounits"],\n            text=True,\n            stderr=subprocess.STDOUT,\n            timeout=15,\n        ).strip()\n    except (OSError, subprocess.SubprocessError):\n        return {"present": False, "name": "cpu", "compute_cap": "", "cuda_arch": "", "vram_mib": 0}\n    first = out.splitlines()[0] if out else ""\n    parts = [p.strip() for p in first.split(",")]\n    if len(parts) < 3:\n        return {"present": False, "name": "cpu", "compute_cap": "", "cuda_arch": "", "vram_mib": 0}\n    name, cap, mem_str = parts[0], parts[1], parts[2]\n    cuda_arch = re.sub(r"[^0-9]", "", cap)\n    try:\n        vram_mib = int(float(mem_str))\n    except ValueError:\n        vram_mib = 0\n    return {\n        "present": vram_mib > 0 and bool(cuda_arch),\n        "name": name,\n        "compute_cap": cap,\n        "cuda_arch": cuda_arch,\n        "vram_mib": vram_mib,\n    }\n\n\ndef derive_shard6_link_name(first_shard_relpath: str) -> str:\n    """Given `.../Model-00001-of-00006.gguf`, return `.../Model-00006-of-00006.gguf`."""\n    match = _SHARD_FIRST_RE.match(first_shard_relpath)\n    if not match:\n        raise ValueError(f"Expected split GGUF shard 1 matching \'*-00001-of-0000N.gguf\', got {first_shard_relpath!r}")\n    prefix, total_str = match.group(1), match.group(2)\n    width = len(total_str)\n    total = int(total_str)\n    return f"{prefix}-{total:0{width}d}-of-{total:0{width}d}.gguf"\n\n\ndef ensure_engram_shard_symlink(local_dir: Path, shard6_relpath: str, engram_relpath: str) -> Path:\n    """Idempotently link the standalone Engram GGUF file to shard N so `llama.cpp` loads all shards."""\n    link_path = local_dir / shard6_relpath\n    engram_path = local_dir / engram_relpath\n    if not engram_path.is_file():\n        raise FileNotFoundError(f"Engram shard file does not exist: {engram_path}")\n    link_path.parent.mkdir(parents=True, exist_ok=True)\n    rel_target = os.path.relpath(engram_path, start=link_path.parent)\n    if link_path.is_symlink():\n        if os.readlink(link_path) == rel_target and link_path.is_file():\n            return link_path\n        link_path.unlink()\n    elif link_path.exists():\n        if link_path.stat().st_size == engram_path.stat().st_size:\n            return link_path\n        link_path.unlink()\n    link_path.symlink_to(rel_target)\n    return link_path\n\n\ndef default_cpu_threads(cpu_count: int | None = None) -> tuple[int, int]:\n    """Return `(threads_decode, threads_batch)` tuned for CPU-side `engram_hash` + `IQ4_NL` row gathers."""\n    logical = max(2, int(cpu_count or os.cpu_count() or 8))\n    threads_decode = max(4, min(16, logical // 2)) if logical >= 8 else logical\n    threads_batch = max(threads_decode, min(32, logical))\n    return threads_decode, threads_batch\n\n\ndef normalize_reasoning_effort(raw: str | int | None) -> str | None:\n    """Validate optional CLI/API `reasoning_effort` (`minimal|low|medium|high|xhigh|max` or `1..100`)."""\n    if raw is None:\n        return None\n    text = str(raw).strip().lower()\n    if not text or text == "default":\n        return None\n    if text in REASONING_EFFORT_LEVELS:\n        return text\n    try:\n        val = int(text)\n    except ValueError as err:\n        allowed = ", ".join(REASONING_EFFORT_LEVELS)\n        raise ValueError(f"reasoning_effort must be in 1..100 or one of {allowed} (got {raw!r})") from err\n    if val < 1 or val > 100:\n        raise ValueError(f"reasoning_effort integer must be in 1..100, got {val}")\n    return str(val)\n\n\ndef build_server_args(\n    *,\n    server_bin: Path | str,\n    model_path: Path | str,\n    alias: str,\n    port: int,\n    key_file: Path | str,\n    plan: dict[str, Any],\n    projector_path: Path | str | None = None,\n    chat_template_file: Path | str | None = None,\n    reasoning: str = "on",\n    reasoning_effort: str | int | None = None,\n    threads: int | None = None,\n    threads_batch: int | None = None,\n) -> list[str]:\n    """Construct exact loopback `llama-server` command arguments for DeepSeek-V4.1-Flash Coder."""\n    engram_in_ram = bool(plan.get("engram_in_ram", False))\n    load_mode = "none" if engram_in_ram else "mmap"\n    lazy_mode = "off" if engram_in_ram else "on"\n    t_dec_default, t_bat_default = default_cpu_threads()\n    t_dec = int(threads) if threads is not None else t_dec_default\n    t_bat = int(threads_batch) if threads_batch is not None else t_bat_default\n\n    args = [\n        str(server_bin),\n        "-m", str(model_path),\n        "--alias", str(alias),\n        "--host", "127.0.0.1",\n        "--port", str(int(port)),\n        "-ngl", str(plan["gpu_layers"]),\n        "-c", str(int(plan["context"])),\n        "-np", str(int(plan["parallel"])),\n        "--load-mode", load_mode,\n        "--lazy-mode", lazy_mode,\n        "--override-tensor", "per_layer_token_embd.weight=CPU",\n        "-t", str(t_dec),\n        "-tb", str(t_bat),\n        "-ctk", str(plan["cache_type"]),\n        "-ctv", str(plan["cache_type"]),\n        "-fa", "on",\n        "-b", str(int(plan["batch"])),\n        "-ub", str(int(plan["ubatch"])),\n        "--jinja",\n    ]\n    if chat_template_file is not None:\n        args += ["--chat-template-file", str(chat_template_file)]\n    args += [\n        "--reasoning-format", "deepseek",\n        "--api-key-file", str(key_file),\n        "--no-context-shift",\n        # Multi-turn agent prefix reuse: more SWA/compressed-KV checkpoints per slot + host prompt cache\n        "--ctx-checkpoints", "32",\n        "--cache-ram", "16384" if engram_in_ram else "4096",\n    ]\n    if engram_in_ram:\n        # Without this, CPU-resident tensors land in a CUDA_Host (cudaMallocHost) pinned buffer, i.e. the\n        # 103 GiB Engram table would be page-locked; plain malloc\'d CPU buffers are faster to load and safer.\n        args += ["--no-host"]\n    if int(plan.get("n_cpu_moe") or 0) > 0:\n        args += ["--n-cpu-moe", str(int(plan["n_cpu_moe"]))]\n    if projector_path is not None and plan.get("enable_vision", True):\n        args += ["--mmproj", str(projector_path)]\n    reasoning_mode = (reasoning or "").strip().lower()\n    if reasoning_mode in ("on", "off", "auto"):\n        args += ["--reasoning", reasoning_mode]\n    effort_norm = normalize_reasoning_effort(reasoning_effort)\n    if effort_norm is not None:\n        args += ["--reasoning-effort", effort_norm]\n    return args\n\n\n_GGUF_SCALAR_SIZES: dict[int, int] = {\n    0: 1,   # UINT8\n    1: 1,   # INT8\n    2: 2,   # UINT16\n    3: 2,   # INT16\n    4: 4,   # UINT32\n    5: 4,   # INT32\n    6: 4,   # FLOAT32\n    7: 1,   # BOOL\n    10: 8,  # UINT64\n    11: 8,  # INT64\n    12: 8,  # FLOAT64\n}\n\n\ndef extract_gguf_chat_template(gguf_path: Path | str) -> str | None:\n    """Read `tokenizer.chat_template` directly from a GGUF v2/v3 header using stdlib `struct`."""\n    path = Path(gguf_path)\n    if not path.is_file():\n        return None\n    try:\n        with path.open("rb") as f:\n            hdr = f.read(24)\n            if len(hdr) < 24 or hdr[:4] != b"GGUF":\n                return None\n            version, _n_tensors, n_kv = struct.unpack("<IQQ", hdr[4:24])\n            if version not in (2, 3) or n_kv > 100_000:\n                return None\n\n            def _read_exact(n: int) -> bytes:\n                buf = f.read(n)\n                if len(buf) != n:\n                    raise EOFError("unexpected EOF in GGUF metadata")\n                return buf\n\n            def _skip_value(vtype: int) -> None:\n                fixed = _GGUF_SCALAR_SIZES.get(vtype)\n                if fixed is not None:\n                    f.seek(fixed, os.SEEK_CUR)\n                    return\n                if vtype == 8:  # STRING\n                    (slen,) = struct.unpack("<Q", _read_exact(8))\n                    f.seek(slen, os.SEEK_CUR)\n                    return\n                if vtype == 9:  # ARRAY\n                    elem_type, count = struct.unpack("<IQ", _read_exact(12))\n                    elem_fixed = _GGUF_SCALAR_SIZES.get(elem_type)\n                    if elem_fixed is not None:\n                        f.seek(elem_fixed * count, os.SEEK_CUR)\n                    else:\n                        for _ in range(count):\n                            _skip_value(elem_type)\n                    return\n                raise ValueError(f"unknown GGUF value type {vtype}")\n\n            for _ in range(n_kv):\n                (klen,) = struct.unpack("<Q", _read_exact(8))\n                key = _read_exact(klen).decode("utf-8", errors="replace")\n                (vtype,) = struct.unpack("<I", _read_exact(4))\n                if key == "tokenizer.chat_template" and vtype == 8:\n                    (slen,) = struct.unpack("<Q", _read_exact(8))\n                    return _read_exact(slen).decode("utf-8")\n                _skip_value(vtype)\n    except (OSError, EOFError, ValueError, struct.error):\n        return None\n    return None\n\n\ndef patch_deepseek_chat_template(template: str) -> str:\n    """Expand DeepSeek-V4.1 `effort_map` for full OpenAI `reasoning_effort` compatibility & calibrated default `50`.\n\n    Stock template only accepts `"low"`, `"high"`, `"max"` or `1..100` (raising a Jinja exception when\n    OpenAI SDKs / Coding Agents pass `"medium"`, `"minimal"`, or `"xhigh"`), and defaults to `"high"` (75)\n    instead of GSQ-RCO-Coder\'s calibrated `50`. Keeps `enable_thinking` and the full-width-bar `DSML` token structure intact so\n    `llama-server`\'s differential autoparser and DeepSeek V4.1 tool parser work identically.\n    """\n    patched = template.replace(\n        \'{%- if reasoning_effort is not defined or reasoning_effort is none -%}{%- set reasoning_effort = "high" -%}{%- endif -%}\',\n        \'{%- if reasoning_effort is not defined or reasoning_effort is none -%}{%- set reasoning_effort = 50 -%}{%- endif -%}\',\n    )\n    patched = patched.replace(\n        \'{%- set effort_map = {"low": 50, "high": 75, "max": 100} -%}\',\n        \'{%- set effort_map = {"minimal": 25, "low": 50, "medium": 60, "high": 75, "xhigh": 90, "max": 100} -%}\',\n    )\n    patched = patched.replace(\n        \'one of \\\\"low\\\\", \\\\"high\\\\", \\\\"max\\\\"\',\n        \'one of \\\\"minimal\\\\", \\\\"low\\\\", \\\\"medium\\\\", \\\\"high\\\\", \\\\"xhigh\\\\", \\\\"max\\\\"\',\n    )\n    return patched\n\n\ndef prepare_patched_chat_template(gguf_path: Path | str, out_path: Path | str) -> Path | None:\n    """Extract & patch `tokenizer.chat_template` from `gguf_path`, writing `out_path` if present."""\n    raw = extract_gguf_chat_template(gguf_path)\n    if not raw:\n        return None\n    patched = patch_deepseek_chat_template(raw)\n    target = Path(out_path)\n    target.parent.mkdir(parents=True, exist_ok=True)\n    target.write_text(patched, encoding="utf-8")\n    return target\n\n\ndef should_evict_page_cache(plan: dict[str, Any]) -> bool:\n    """Only safe when no weights are served from mmap\'d trunk shards (CPU MoE layers would page-fault)."""\n    return int(plan.get("n_cpu_moe") or 0) == 0 and plan.get("gpu_layers") == "all"\n\n\ndef evict_gguf_page_cache(\n    model_dir: Path | str,\n    *,\n    keep_engram: bool = False,\n    engram_filename: str = ENGRAM_FILENAME,\n    log=print,\n) -> int:\n    """Drop reclaimable Linux Page Cache for `.gguf` files via `POSIX_FADV_DONTNEED` after weight load.\n\n    When `engram_in_ram=True` (`--load-mode none --lazy-mode off`), all GPU trunk weights reside in VRAM\n    and Engram resides in anonymous CPU RAM (`folio_mapped == false`), so evicting `.gguf` Page Cache\n    reclaims ~68–170 GiB of duplicate file cache cleanly.\n    """\n    fadvise = getattr(os, "posix_fadvise", None)\n    dontneed = getattr(os, "POSIX_FADV_DONTNEED", None)\n    root = Path(model_dir)\n    if fadvise is None or dontneed is None or not root.is_dir():\n        return 0\n    engram_resolved = (root / engram_filename).resolve() if keep_engram else None\n    reclaimed_files = 0\n    reclaimed_bytes = 0\n    for candidate in sorted(root.rglob("*.gguf")):\n        if candidate.is_symlink() or not candidate.is_file():\n            continue\n        if engram_resolved is not None and candidate.resolve() == engram_resolved:\n            continue\n        try:\n            fd = os.open(candidate, os.O_RDONLY)\n            try:\n                size = os.fstat(fd).st_size\n                fadvise(fd, 0, 0, dontneed)\n                reclaimed_files += 1\n                reclaimed_bytes += size\n            finally:\n                os.close(fd)\n        except OSError:\n            continue\n    if reclaimed_files:\n        log(f"Released Linux Page Cache hints on {reclaimed_files} GGUF shard(s) ({reclaimed_bytes / (1024**3):.1f} GiB)")\n    return reclaimed_files\n\n\ndef looks_like_oom(log_text: str) -> bool:\n    """Return True when `llama-server` stderr/stdout indicates a CUDA/host memory allocation failure."""\n    lowered = (log_text or "").lower()\n    return any(marker in lowered for marker in OOM_LOG_MARKERS)\n\n\n# --------------------------------------------------------------------------- Colab VM staging helpers\ndef write_json_atomic(path: Path | str, value: Any) -> None:\n    target = Path(path)\n    target.parent.mkdir(parents=True, exist_ok=True)\n    tmp = target.with_name(f".{target.name}.{os.getpid()}.tmp")\n    tmp.write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\\n", encoding="utf-8")\n    tmp.replace(target)\n\n\ndef digest_file(path: Path | str) -> str:\n    h = hashlib.sha256()\n    with open(path, "rb") as handle:\n        while chunk := handle.read(16 * 1024 * 1024):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef require_space(path: Path | str, needed_bytes: int) -> None:\n    target = Path(path)\n    target.mkdir(parents=True, exist_ok=True)\n    free = shutil.disk_usage(target).free\n    if free < needed_bytes:\n        raise RuntimeError(f"{target}: need {needed_bytes / 1e9:.1f} GB free on local SSD, have {free / 1e9:.1f} GB")\n\n\ndef _cmd_out(command: list[str], *, env: dict[str, str] | None = None, timeout: int = 60) -> str:\n    return subprocess.check_output(command, text=True, stderr=subprocess.STDOUT, env=env, timeout=timeout).strip()\n\n\ndef _run_logged(command: list[str], log_file: Path, *, env: dict[str, str] | None = None, timeout: int = 3600) -> None:\n    log_file.parent.mkdir(parents=True, exist_ok=True)\n    with log_file.open("a", encoding="utf-8") as handle:\n        res = subprocess.run(command, stdout=handle, stderr=subprocess.STDOUT, env=env, timeout=timeout)\n    if res.returncode:\n        raise RuntimeError(f"Command failed ({res.returncode}): {\' \'.join(command[:4])} (see {log_file})")\n\n\ndef _restore_valid_bin(bin_dir: Path, env: dict[str, str], *, expect_cuda: bool) -> bool:\n    server = bin_dir / "llama-server"\n    if not server.is_file():\n        return False\n    try:\n        server.chmod(0o755)\n        _cmd_out([str(server), "--version"], env=env, timeout=30)\n        if expect_cuda:\n            devices = _cmd_out([str(server), "--list-devices"], env=env, timeout=30)\n            if "CUDA" not in devices:\n                return False\n        help_text = _cmd_out([str(server), "--help"], env=env, timeout=30)\n        return all(flag in help_text for flag in ("--lazy-mode", "--load-mode", "--api-key-file", "--n-cpu-moe"))\n    except (OSError, subprocess.SubprocessError):\n        return False\n\n\ndef prepare_llama_runtime(\n    *,\n    work_dir: Path,\n    cache_root: Path,\n    llama_repo: str = DEFAULT_LLAMA_REPO,\n    llama_commit: str = DEFAULT_LLAMA_COMMIT,\n    gpu_info: dict[str, Any] | None = None,\n    log=print,\n) -> tuple[Path, dict[str, str], dict[str, Any]]:\n    """Restore or compile pinned `llama-server` for the active GPU\'s CUDA architecture."""\n    gpu = gpu_info or detect_gpu_info()\n    has_cuda = bool(gpu.get("present") and gpu.get("cuda_arch"))\n    cuda_arch = gpu["cuda_arch"] if has_cuda else "cpu"\n    nvcc_ver = _cmd_out(["nvcc", "--version"], timeout=30) if has_cuda else "no-cuda"\n    os_rel = Path("/etc/os-release").read_text(encoding="utf-8", errors="replace") if Path("/etc/os-release").is_file() else platform.platform()\n    signature = {\n        "llama_commit": llama_commit,\n        "cuda_arch": cuda_arch,\n        "cuda": nvcc_ver,\n        "os_release": os_rel,\n        "arch": platform.machine(),\n        "glibc": list(platform.libc_ver()),\n        "targets": ["llama-server"],\n    }\n    key = hashlib.sha256(json.dumps(signature, sort_keys=True).encode()).hexdigest()[:20]\n    cache_dir = cache_root / f"sm{cuda_arch}-{key}"\n    cache_dir.mkdir(parents=True, exist_ok=True)\n\n    bin_dir = work_dir / "runtime" / "bin"\n    bin_dir.mkdir(parents=True, exist_ok=True)\n    env = dict(os.environ, LD_LIBRARY_PATH=str(bin_dir) + ":" + os.environ.get("LD_LIBRARY_PATH", ""))\n    local_manifest = work_dir / "runtime-manifest.json"\n    if bin_dir.exists() and any(bin_dir.iterdir()):\n        try:\n            cached_sig = json.loads(local_manifest.read_text(encoding="utf-8")) if local_manifest.is_file() else None\n        except (OSError, ValueError):\n            cached_sig = None\n        if cached_sig != signature:\n            shutil.rmtree(bin_dir)\n            bin_dir.mkdir(parents=True, exist_ok=True)\n\n    archive = cache_dir / "runtime.tar"\n    checksum = cache_dir / "runtime.sha256"\n    if not _restore_valid_bin(bin_dir, env, expect_cuda=has_cuda) and archive.is_file() and checksum.is_file():\n        log(f"Restoring prebuilt llama.cpp runtime archive from Drive: {cache_dir}")\n        local_tar = work_dir / "runtime.tar"\n        shutil.copyfile(archive, local_tar)\n        if digest_file(local_tar) != checksum.read_text(encoding="utf-8").strip():\n            local_tar.unlink(missing_ok=True)\n            raise RuntimeError("Drive runtime cache SHA256 mismatch; refusing corrupt archive")\n        with tarfile.open(local_tar) as bundle:\n            bundle.extractall(work_dir / "runtime", filter="data")\n        local_tar.unlink(missing_ok=True)\n\n    if not _restore_valid_bin(bin_dir, env, expect_cuda=has_cuda):\n        build_log = work_dir / "build.log"\n        log(f"Building llama.cpp ({llama_commit[:10]}) for target={cuda_arch}; log={build_log}")\n        source = work_dir / "llama.cpp-src"\n        if (source / ".git").exists():\n            shutil.rmtree(source)\n        _run_logged(["git", "clone", "--no-checkout", "--filter=blob:none", llama_repo, str(source)], build_log)\n        _run_logged(["git", "-C", str(source), "fetch", "--depth", "1", "origin", llama_commit], build_log)\n        _run_logged(["git", "-C", str(source), "checkout", "--detach", llama_commit], build_log)\n        cmake_args = [\n            "cmake", "-S", str(source), "-B", str(source / "build"),\n            "-DGGML_NATIVE=OFF", "-DCMAKE_BUILD_TYPE=Release",\n            "-DLLAMA_CURL=OFF", "-DLLAMA_OPENSSL=OFF", "-DLLAMA_BUILD_TESTS=OFF",\n            "-DLLAMA_BUILD_EXAMPLES=OFF",\n        ]\n        if has_cuda:\n            cmake_args += ["-DGGML_CUDA=ON", f"-DCMAKE_CUDA_ARCHITECTURES={cuda_arch}"]\n        else:\n            cmake_args += ["-DGGML_CUDA=OFF"]\n        _run_logged(cmake_args, build_log)\n        jobs = str(min(12, os.cpu_count() or 4))\n        _run_logged(\n            ["cmake", "--build", str(source / "build"), "--config", "Release", "-j", jobs, "--target", "llama-server"],\n            build_log,\n        )\n        shutil.copytree(source / "build" / "bin", bin_dir, dirs_exist_ok=True, symlinks=False)\n        for lib_candidate in (source / "build").rglob("*.so*"):\n            if lib_candidate.is_file() and not (bin_dir / lib_candidate.name).exists():\n                shutil.copy2(lib_candidate, bin_dir / lib_candidate.name)\n        # Reclaim ~4-6 GB CUDA build artifacts immediately so local SSD has full room for 183.7 GB weights\n        shutil.rmtree(source, ignore_errors=True)\n        if not _restore_valid_bin(bin_dir, env, expect_cuda=has_cuda):\n            raise RuntimeError(f"Compiled llama-server failed validation (expect_cuda={has_cuda})")\n        local_tar = work_dir / "runtime.tar"\n        with tarfile.open(local_tar, "w", dereference=True) as bundle:\n            bundle.add(bin_dir, arcname="bin")\n        sha = digest_file(local_tar)\n        partial = cache_dir / f".runtime.tar.{os.getpid()}.partial"\n        shutil.copyfile(local_tar, partial)\n        partial.replace(archive)\n        checksum.write_text(sha + "\\n", encoding="utf-8")\n        write_json_atomic(cache_dir / "manifest.json", signature)\n        local_tar.unlink(missing_ok=True)\n        log(f"Archived llama-server runtime to {cache_dir}")\n\n    write_json_atomic(local_manifest, signature)\n    return bin_dir, env, gpu\n\n\ndef _hf_client(work_dir: Path):\n    os.environ["HF_XET_CHUNK_CACHE_SIZE_BYTES"] = "0"\n    os.environ.setdefault("HF_XET_HIGH_PERFORMANCE", "1")\n    try:\n        import hf_xet  # noqa: F401\n        from huggingface_hub import HfApi, hf_hub_download\n    except ImportError:\n        _run_logged([sys.executable, "-m", "pip", "install", "-U", "huggingface_hub", "hf_xet"], work_dir / "download.log", timeout=600)\n        from huggingface_hub import HfApi, hf_hub_download\n    return HfApi, hf_hub_download\n\n\ndef _resolve_model_lock(repo: str, quant: str, service_dir: Path, work_dir: Path) -> dict[str, Any]:\n    safe_repo = re.sub(r"[^A-Za-z0-9._-]+", "_", repo)\n    lock_path = service_dir / f"model-lock-{safe_repo}-{quant}.json"\n    if lock_path.is_file():\n        lock = json.loads(lock_path.read_text(encoding="utf-8"))\n        if lock.get("repo") == repo:\n            return lock\n    if repo == DEFAULT_REPO and quant.upper() == DEFAULT_QUANT:\n        write_json_atomic(lock_path, DEFAULT_MODEL_LOCK)\n        return DEFAULT_MODEL_LOCK\n\n    HfApi, _ = _hf_client(work_dir)\n    info = HfApi().model_info(repo, files_metadata=True)\n    trunk_files = []\n    engram_entry = None\n    mmproj_entry = None\n    q_upper = quant.upper()\n    for entry in info.siblings:\n        rfn = entry.rfilename\n        if not rfn.lower().endswith(".gguf"):\n            continue\n        if not entry.lfs:\n            raise RuntimeError(f"Expected LFS SHA256 metadata for {rfn}")\n        rec = {"name": rfn, "size": entry.size, "sha256": entry.lfs.sha256}\n        lower = rfn.lower()\n        if "mmproj" in lower:\n            mmproj_entry = rec\n        elif "engram" in lower:\n            engram_entry = rec\n        elif q_upper in rfn.upper():\n            trunk_files.append(rec)\n    trunk_files.sort(key=lambda item: item["name"])\n    if not trunk_files:\n        raise RuntimeError(f"No GGUF shards matching quant {quant!r} found in {repo}")\n    shard6_link = None\n    first_name = trunk_files[0]["name"]\n    if _SHARD_FIRST_RE.match(first_name):\n        expected_last = derive_shard6_link_name(first_name)\n        if not any(item["name"] == expected_last for item in trunk_files):\n            if engram_entry is None:\n                raise RuntimeError(f"Split GGUF {first_name} is missing final shard {expected_last} and no Engram GGUF found")\n            trunk_files.append(engram_entry)\n            shard6_link = expected_last\n    lock = {\n        "repo": repo,\n        "revision": info.sha,\n        "shard6_link": shard6_link,\n        "engram_file": engram_entry["name"] if engram_entry else None,\n        "files": trunk_files,\n        "mmproj": mmproj_entry,\n    }\n    write_json_atomic(lock_path, lock)\n    return lock\n\n\ndef _xet_available() -> bool:\n    try:\n        import hf_xet  # noqa: F401\n    except ImportError:\n        return False\n    return True\n\n\ndef _stage_one_file(\n    item: dict[str, Any],\n    *,\n    repo: str,\n    revision: str,\n    local_dir: Path,\n    drive_backup_root: Path | None,\n    download_fn,\n    verified_records: dict[str, dict[str, Any]],\n    log=print,\n) -> Path:\n    target = local_dir / item["name"]\n    target.parent.mkdir(parents=True, exist_ok=True)\n    prev = verified_records.get(item["name"])\n    if (\n        target.is_file()\n        and target.stat().st_size == item["size"]\n        and prev\n        and prev.get("sha256") == item["sha256"]\n        and prev.get("mtime_ns") == target.stat().st_mtime_ns\n    ):\n        return target\n    if target.is_file() and target.stat().st_size == item["size"] and digest_file(target) == item["sha256"]:\n        return target\n\n    if download_fn is not None:\n        log(f"Downloading {item[\'name\']} ({item[\'size\'] / 1e9:.2f} GB) -> local SSD")\n        file_tag = hashlib.sha256(item["name"].encode("utf-8")).hexdigest()[:12]\n        scratch_dir = local_dir / f".hf-stage-{os.getpid()}-{file_tag}"\n        shutil.rmtree(scratch_dir, ignore_errors=True)\n        scratch_dir.mkdir(parents=True, exist_ok=True)\n        try:\n            downloaded = Path(\n                download_fn(\n                    repo_id=repo,\n                    filename=item["name"],\n                    revision=revision,\n                    local_dir=str(scratch_dir),\n                )\n            )\n            if not downloaded.is_file() or downloaded.stat().st_size != item["size"]:\n                downloaded.unlink(missing_ok=True)\n                raise RuntimeError(f"Downloaded file size mismatch: {item[\'name\']}")\n            # hf_xet already verifies every chunk against its content-addressed Merkle hash during reconstruction;\n            # a second full-file SHA256 pass costs ~5 min on the 110.6 GB Engram file, so it is opt-in.\n            strict = os.environ.get("CT_DS_VERIFY_SHA256", "0") == "1" or not _xet_available()\n            if strict and digest_file(downloaded) != item["sha256"]:\n                downloaded.unlink(missing_ok=True)\n                raise RuntimeError(f"Downloaded file SHA256 mismatch: {item[\'name\']}")\n            target.parent.mkdir(parents=True, exist_ok=True)\n            downloaded.replace(target)\n        finally:\n            shutil.rmtree(scratch_dir, ignore_errors=True)\n        log(f"Verified {\'SHA256\' if strict else \'xet chunk hashes + size\'} ({item[\'size\'] / 1e9:.2f} GB): {item[\'name\']}")\n        return target\n\n    if drive_backup_root is not None:\n        backup = drive_backup_root / repo / item["name"]\n        if backup.is_file() and backup.stat().st_size == item["size"]:\n            log(f"Copying Drive backup -> local SSD: {item[\'name\']}")\n            partial = target.with_name(f".{target.name}.{os.getpid()}.partial")\n            shutil.copyfile(backup, partial)\n            if partial.stat().st_size != item["size"] or digest_file(partial) != item["sha256"]:\n                partial.unlink(missing_ok=True)\n                raise RuntimeError(f"Drive backup SHA256 mismatch: {item[\'name\']}")\n            partial.replace(target)\n            return target\n    raise RuntimeError(f"Unable to stage {item[\'name\']} from Hugging Face or Drive backup")\n\n\ndef stage_model_and_projector(\n    *,\n    repo: str,\n    quant: str,\n    enable_vision: bool,\n    service_dir: Path,\n    work_dir: Path,\n    drive_backup_root: Path | None = None,\n    log=print,\n) -> tuple[Path, Path | None]:\n    """Download/verify all trunk shards + Engram shard-6 symlink + optional BF16 mmproj on local NVMe SSD."""\n    os.environ["HF_XET_CHUNK_CACHE_SIZE_BYTES"] = "0"\n    lock = _resolve_model_lock(repo, quant, service_dir, work_dir)\n    local_dir = work_dir / "model"\n    local_dir.mkdir(parents=True, exist_ok=True)\n    verified_path = local_dir / "verified.json"\n    try:\n        verified_records: dict[str, dict[str, Any]] = (\n            json.loads(verified_path.read_text(encoding="utf-8")) if verified_path.is_file() else {}\n        )\n    except (OSError, ValueError):\n        verified_records = {}\n\n    items = list(lock["files"])\n    mmproj_spec: dict[str, Any] | None = None\n    if enable_vision:\n        if repo == DEFAULT_REPO and lock["revision"] == DEFAULT_REVISION:\n            mmproj_spec = VISION_PROJECTOR_SPEC\n        else:\n            mmproj_spec = lock.get("mmproj")\n        if mmproj_spec is not None:\n            items.append(mmproj_spec)\n\n    missing_bytes = 0\n    for item in items:\n        target = local_dir / item["name"]\n        if not (target.is_file() and target.stat().st_size == item["size"]):\n            missing_bytes += int(item["size"])\n    if missing_bytes > 0:\n        # Reserve 6 GiB headroom above missing bytes while respecting Colab\'s ~253 GB local NVMe SSD\n        require_space(work_dir, missing_bytes + 6 * 1024**3)\n\n    download_fn = None\n    if missing_bytes > 0 or not verified_records:\n        try:\n            _, download_fn = _hf_client(work_dir)\n        except Exception as err:\n            log(f"HF client init warning ({type(err).__name__}: {err}); falling back to local/Drive files")\n\n    # Order largest-first across 2 isolated workers so the 110.6 GB Engram shard overlaps the 73.1 GB trunk shards\n    ordered_items = sorted(items, key=lambda it: int(it["size"]), reverse=True)\n    if ordered_items:\n        with ThreadPoolExecutor(max_workers=2) as pool:\n            futures = [\n                pool.submit(\n                    _stage_one_file,\n                    it,\n                    repo=repo,\n                    revision=lock["revision"],\n                    local_dir=local_dir,\n                    drive_backup_root=drive_backup_root,\n                    download_fn=download_fn,\n                    verified_records=verified_records,\n                    log=log,\n                )\n                for it in ordered_items\n            ]\n            for fut in futures:\n                fut.result()\n\n    # Purge transient HF cache directories so local NVMe SSD retains maximum free space\n    shutil.rmtree(local_dir / ".cache", ignore_errors=True)\n    shutil.rmtree(Path.home() / ".cache" / "huggingface" / "xet", ignore_errors=True)\n\n    if lock.get("shard6_link") and lock.get("engram_file"):\n        link = ensure_engram_shard_symlink(local_dir, lock["shard6_link"], lock["engram_file"])\n        log(f"Engram shard symlink ready: {link.name} -> {lock[\'engram_file\']}")\n\n    updated_records = {}\n    for it in items:\n        p = local_dir / it["name"]\n        updated_records[it["name"]] = {\n            "size": it["size"],\n            "sha256": it["sha256"],\n            "mtime_ns": p.stat().st_mtime_ns,\n        }\n    write_json_atomic(verified_path, updated_records)\n\n    first_shard = local_dir / lock["files"][0]["name"]\n    projector_path = (local_dir / mmproj_spec["name"]) if mmproj_spec is not None else None\n    return first_shard, projector_path\n', encoding="utf-8")
if str(DS_WORK) not in sys.path:
    sys.path.insert(0, str(DS_WORK))

from deepseek_coder_runtime import (
    DEFAULT_ALIAS,
    DEFAULT_LLAMA_COMMIT,
    DEFAULT_LLAMA_REPO,
    DEFAULT_QUANT,
    DEFAULT_REPO,
    build_server_args,
    detect_gpu_info,
    detect_host_ram_mib,
    evict_gguf_page_cache,
    looks_like_oom,
    plan_hardware_config,
    prepare_llama_runtime,
    prepare_patched_chat_template,
    should_evict_page_cache,
    stage_model_and_projector,
    write_json_atomic,
)

def dslog(msg):
    print(time.strftime("[%F %T] [deepseek-coder] ") + msg, flush=True)

if MOUNT_DRIVE and not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Drive is not mounted. Run via `colabllm deepseek-coder up`.")

DS_SERVICE_DIR = Path(DATA_DIR)
DS_SERVICE_DIR.mkdir(parents=True, exist_ok=True)
DS_DRIVE_BACKUP_ROOT = (
    Path("/content/drive/MyDrive/models") if (MOUNT_DRIVE and Path("/content/drive/MyDrive").is_dir()) else None
)
DS_RUNTIME_CACHE = DS_SERVICE_DIR / "runtime-cache"

DS_CFG = {
    "repo": os.environ.get("CT_DS_REPO", DEFAULT_REPO),
    "quant": os.environ.get("CT_DS_QUANT", DEFAULT_QUANT),
    "alias": os.environ.get("CT_DS_ALIAS", DEFAULT_ALIAS),
    "context": os.environ.get("CT_DS_CONTEXT", "auto"),
    "parallel": int(os.environ.get("CT_DS_PARALLEL", "1")),
    "cache_type": os.environ.get("CT_DS_CACHE_TYPE", "f16"),
    "batch": int(os.environ.get("CT_DS_BATCH", "4096")),
    "ubatch": os.environ.get("CT_DS_UBATCH", "auto"),
    "cpu_moe": os.environ.get("CT_DS_CPU_MOE", "auto"),
    "reasoning": os.environ.get("CT_DS_REASONING", "on"),
    "reasoning_effort": os.environ.get("CT_DS_REASONING_EFFORT", ""),
    "enable_vision": os.environ.get("CT_DS_VISION", "1") != "0",
    "llama_repo": os.environ.get("CT_LLAMA_REPO", DEFAULT_LLAMA_REPO),
    "llama_commit": os.environ.get("CT_LLAMA_COMMIT", DEFAULT_LLAMA_COMMIT),
}

DS_GPU = detect_gpu_info()
DS_RAM_MIB = detect_host_ram_mib()
HW_PLAN = plan_hardware_config(
    vram_mib=DS_GPU["vram_mib"],
    ram_mib=DS_RAM_MIB,
    requested_context=DS_CFG["context"],
    parallel=DS_CFG["parallel"],
    cache_type=DS_CFG["cache_type"],
    enable_vision=DS_CFG["enable_vision"],
    batch=DS_CFG["batch"],
    requested_ubatch=DS_CFG["ubatch"],
    requested_cpu_moe=DS_CFG["cpu_moe"],
)
dslog(
    f"Hardware plan: gpu={DS_GPU['name']} ({DS_GPU['vram_mib']} MiB VRAM, sm_{DS_GPU['cuda_arch'] or 'cpu'}) "
    f"ram={DS_RAM_MIB} MiB -> mode={HW_PLAN['mode']} context={HW_PLAN['context']} "
    f"(kv={HW_PLAN['kv_cache_mib']} MiB, ubatch={HW_PLAN['ubatch']}, n_cpu_moe={HW_PLAN['n_cpu_moe']}, "
    f"engram_in_ram={HW_PLAN['engram_in_ram']})"
)

with ThreadPoolExecutor(max_workers=2) as _pool:
    _f_runtime = _pool.submit(
        prepare_llama_runtime,
        work_dir=DS_WORK,
        cache_root=DS_RUNTIME_CACHE,
        llama_repo=DS_CFG["llama_repo"],
        llama_commit=DS_CFG["llama_commit"],
        gpu_info=DS_GPU,
        log=dslog,
    )
    _f_model = _pool.submit(
        stage_model_and_projector,
        repo=DS_CFG["repo"],
        quant=DS_CFG["quant"],
        enable_vision=DS_CFG["enable_vision"],
        service_dir=DS_SERVICE_DIR,
        work_dir=DS_WORK,
        drive_backup_root=DS_DRIVE_BACKUP_ROOT,
        log=dslog,
    )
    while not (_f_runtime.done() and _f_model.done()):
        for _fut in (_f_runtime, _f_model):
            if _fut.done() and _fut.exception():
                raise _fut.exception()
        _free_gb = shutil.disk_usage(DS_WORK).free / 1e9
        dslog(
            f"Setup heartbeat | runtime={'done' if _f_runtime.done() else 'running'} "
            f"model={'done' if _f_model.done() else 'running'} | free_ssd={_free_gb:.1f} GB"
        )
        time.sleep(30)
    BIN_DIR, LLAMA_ENV, _ = _f_runtime.result()
    MODEL_PATH, PROJECTOR_PATH = _f_model.result()

CHAT_TEMPLATE_PATH = prepare_patched_chat_template(MODEL_PATH, DS_WORK / "chat_template.jinja")
dslog(f"Setup complete: model={MODEL_PATH} projector={PROJECTOR_PATH} chat_template={CHAT_TEMPLATE_PATH}")


In [ ]:
# @title 2. Service -- llama-server on 127.0.0.1 (Engram host-RAM or SSD lazy read, hardware-max context, --api-key-file)
KEY_FILE = DS_WORK / "api_key.txt"
KEY_FILE.write_text(API_TOKEN + "\n", encoding="utf-8")
KEY_FILE.chmod(0o600)
SPAWN_LOG_OFFSET = 0

def start_service():
    global SPAWN_LOG_OFFSET
    subprocess.run(["pkill", "-f", str(BIN_DIR / "llama-server")], check=False, timeout=15)
    args = build_server_args(
        server_bin=BIN_DIR / "llama-server",
        model_path=MODEL_PATH,
        alias=DS_CFG["alias"],
        port=SERVICE_PORT,
        key_file=KEY_FILE,
        plan=HW_PLAN,
        projector_path=PROJECTOR_PATH,
        chat_template_file=CHAT_TEMPLATE_PATH,
        reasoning=DS_CFG["reasoning"],
        reasoning_effort=DS_CFG["reasoning_effort"],
    )
    SPAWN_LOG_OFFSET = ct_log_size("llama-server")
    engram_loc = "host-ram" if HW_PLAN.get("engram_in_ram") else "ssd-lazy"
    dslog(
        f"Starting llama-server on 127.0.0.1:{SERVICE_PORT} | ctx={HW_PLAN['context']} "
        f"mode={HW_PLAN['mode']} engram={engram_loc} kv={HW_PLAN['cache_type']} "
        f"batch={HW_PLAN['batch']}/{HW_PLAN['ubatch']} reasoning={DS_CFG['reasoning']}"
    )
    return ct_spawn("llama-server", args, env=LLAMA_ENV)

PROCS["service"] = (start_service(), start_service)


In [ ]:
# @title 3. Auth layer -- Caddy: basic auth for browsers, `Authorization: Bearer <token>` for API clients
# PUBLIC_PORT must be free on the VM: Colab itself uses 8080 (runtime server) and 8012 (colab dfs shim).
# Only used when the service has no auth of its own (CT_AUTH=caddy). The service itself listens on
# 127.0.0.1 only, cloudflared targets Caddy, so every public request passes through auth.
REWRITE_HOST = True   # True: send Host: 127.0.0.1:<port> upstream (Ollama & co. reject foreign Host headers)
import platform
CADDY_ARCH = {"x86_64": "amd64", "aarch64": "arm64"}.get(platform.machine(), "amd64")

def start_caddy():
    if not os.path.exists("/usr/local/bin/caddy"):
        tag = json.loads(urllib.request.urlopen(
            "https://api.github.com/repos/caddyserver/caddy/releases/latest", timeout=30).read())["tag_name"]
        tgz = "/tmp/caddy.tgz"
        with urllib.request.urlopen(f"https://github.com/caddyserver/caddy/releases/download/{tag}/"
                                    f"caddy_{tag.lstrip('v')}_linux_{CADDY_ARCH}.tar.gz", timeout=60) as src, \
                open(tgz, "wb") as dst:
            shutil.copyfileobj(src, dst)
        subprocess.run(["tar", "-xzf", tgz, "-C", "/usr/local/bin", "caddy"], check=True)
    import hashlib
    cookie_tok = hashlib.sha256(f"ct-cookie-v2:{AUTH_USER}:{AUTH_PASS}:{API_TOKEN}".encode()).hexdigest()
    hashed = subprocess.check_output(["caddy", "hash-password", "--plaintext", AUTH_PASS], text=True).strip()
    upstream = f"""reverse_proxy 127.0.0.1:{SERVICE_PORT} {{
            {"header_up Host {upstream_hostport}" if REWRITE_HOST else ""}
            {"header_up Origin http://{upstream_hostport}" if REWRITE_HOST else ""}
            header_up -Authorization
            flush_interval -1
        }}"""
    caddyfile = f"""{{
    admin off
    auto_https off
}}
:{PUBLIC_PORT} {{
    encode zstd gzip
    @bearer header Authorization "Bearer {API_TOKEN}"
    handle @bearer {{
        {upstream}
    }}
    @cookie header Cookie *ct_session={cookie_tok}*
    handle @cookie {{
        {upstream}
    }}
    handle {{
        route {{
            basic_auth {{
                "{AUTH_USER}" {hashed}
            }}
            header +Set-Cookie "ct_session={cookie_tok}; Path=/; HttpOnly; SameSite=Lax"
            {upstream}
        }}
    }}
}}
"""
    open(f"{WORK}/Caddyfile", "w").write(caddyfile)
    subprocess.run(["pkill", "-f", "caddy run"], check=False)
    return ct_spawn("caddy", ["caddy", "run", "--config", f"{WORK}/Caddyfile", "--adapter", "caddyfile"])

AUTH_OK = False   # cell 4 refuses to open the tunnel unless this becomes True (fail closed)
if AUTH == "caddy":
    PROCS["caddy"] = (start_caddy(), start_caddy)
    time.sleep(2)
    if PROCS["caddy"][0].poll() is not None:   # e.g. port taken (Colab uses 8080 itself)
        raise RuntimeError(f"caddy exited -- is :{PUBLIC_PORT} in use?\n" + ct_tail("caddy"))
    # Must be 401 from *our* Caddy and must NOT leak Set-Cookie on unauthenticated 401 responses.
    ct_wait_http(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=60, ok=lambda c: c == 401)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=5)
    except urllib.error.HTTPError as _e:
        if _e.headers.get("Set-Cookie"):
            raise RuntimeError("Caddy leaked Set-Cookie on unauthenticated 401 response!")
    AUTH_OK = True
    print(f"caddy on :{PUBLIC_PORT} -> 127.0.0.1:{SERVICE_PORT} (unauthenticated request -> 401)")
else:
    AUTH_OK = True
    print(f"auth={AUTH}: no proxy, cloudflared targets the service on :{PUBLIC_PORT} directly")


In [ ]:
# @title 4. Tunnel -- Cloudflare by default, or project-configured FRP + HTTPS
"""Self-contained Colab transport cell; injected into every service notebook."""
import hashlib
import json
import os
import platform
import re
import shutil
import signal
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.error
import urllib.request
from pathlib import Path


def render_frpc(config, local_port, ca_path):
    """TOML strings use JSON quoting, never shell interpolation."""
    return f'''serverAddr = {json.dumps(config['server_ip'])}
serverPort = {int(config['server_port'])}
loginFailExit = false
auth.method = "token"
auth.token = {json.dumps(config['token'])}
auth.additionalScopes = {json.dumps(config.get('auth_scopes',['HeartBeats','NewWorkConns']))}
transport.tls.enable = true
transport.tls.trustedCaFile = {json.dumps(str(ca_path))}
transport.tls.serverName = {json.dumps(config.get('server_name') or config['server_ip'])}
log.to = "console"
log.level = "info"
[[proxies]]
name = {json.dumps(config['proxy_name'])}
type = "tcp"
localIP = "127.0.0.1"
localPort = {int(local_port)}
remotePort = {int(config['remote_port'])}
'''


def install_frpc(destination, version):
    if destination.exists():
        return str(destination)
    arch={'x86_64':'amd64','aarch64':'arm64'}.get(platform.machine())
    if not arch:
        raise ValueError('Unsupported FRP client architecture')
    if not re.fullmatch(r'\d+\.\d+\.\d+',version) or tuple(map(int,version.split('.')))<(0,61,0):
        raise ValueError('Unsupported FRP client version; expected stable >= 0.61.0')
    asset=f'frp_{version}_linux_{arch}.tar.gz'
    base=f'https://github.com/fatedier/frp/releases/download/v{version}/'
    with tempfile.TemporaryDirectory() as tmp:
        archive=Path(tmp)/asset
        with urllib.request.urlopen(base+asset,timeout=60) as response, open(archive,'wb') as out:
            shutil.copyfileobj(response,out)
        checks=urllib.request.urlopen(base+'frp_sha256_checksums.txt',timeout=60).read().decode()
        expected=next((line.split()[0] for line in checks.splitlines()
                       if line.split() and line.split()[-1].lstrip('*')==asset),None)
        if expected is None or hashlib.sha256(archive.read_bytes()).hexdigest()!=expected:
            raise ValueError('FRP client checksum verification failed')
        with tarfile.open(archive) as tar:
            entry=tar.getmember(f'frp_{version}_linux_{arch}/frpc')
            if not entry.isfile():
                raise ValueError('Invalid FRP client binary')
            destination.parent.mkdir(parents=True,exist_ok=True)
            with tar.extractfile(entry) as source:
                destination.write_bytes(source.read())
            destination.chmod(0o755)
    return str(destination)


def run_tunnel(ctx):
    if not ctx.get('AUTH_OK'):
        raise RuntimeError('Auth layer not ready; refusing to expose the service')
    work=Path(ctx['WORK'])
    transport=os.environ.get('CT_TRANSPORT','cloudflare')
    if transport not in ('cloudflare','frp'):
        raise ValueError('Unsupported tunnel transport')
    # Stop only the tunnel previously started by this notebook (PID + proc
    # start time prevent accidental termination after PID reuse).
    marker=work/'tunnel-process.json'
    def identity(pid):
        try:
            stat=Path(f'/proc/{pid}/stat').read_text().rsplit(')',1)[1].split()
            return stat[19]
        except OSError:
            return None
    if marker.exists():
        previous=json.loads(marker.read_text())
        if previous.get('start_time') and identity(previous['pid'])==previous['start_time']:
            os.kill(previous['pid'],signal.SIGTERM)
            for _ in range(50):
                if identity(previous['pid']) is None:
                    break
                time.sleep(.1)
    def spawn(name,args):
        (work/(name+'.log')).write_text('')
        proc=ctx['ct_spawn'](name,args)
        marker.write_text(json.dumps(dict(pid=proc.pid,start_time=identity(proc.pid))))
        return proc

    if transport=='frp':
        config={key:os.environ['CT_FRP_'+key.upper()] for key in
                ('server_ip','server_port','remote_port','proxy_name','token')}
        config['server_name']=os.environ.get('CT_FRP_SERVER_NAME') or config['server_ip']
        config['auth_scopes']=json.loads(os.environ.get('CT_FRP_AUTH_SCOPES') or '["HeartBeats","NewWorkConns"]')
        public_host=ctx['PUBLIC_HOST']
        if not public_host:
            raise ValueError('FRP requires a public hostname')
        ca_path=work/'frp-ca.crt'
        ca_path.write_text(os.environ['CT_FRP_CA_CERT'])
        config_path=work/'frpc.toml'
        config_path.touch(mode=0o600,exist_ok=True)
        config_path.chmod(0o600)
        config_path.write_text(render_frpc(config,ctx['PUBLIC_PORT'],ca_path))
        binary=install_frpc(work/('frpc-'+os.environ['CT_FRP_VERSION']),os.environ['CT_FRP_VERSION'])
        result=subprocess.run([binary,'verify','-c',str(config_path)],capture_output=True)
        if result.returncode:
            raise RuntimeError('FRP client configuration validation failed')
        def start():
            return spawn('frpc',[binary,'-c',str(config_path)])
        key='frpc'
    else:
        cf_arch={'x86_64':'amd64','aarch64':'arm64'}.get(platform.machine(),'amd64')
        binary=ctx['ct_fetch_binary']('/usr/local/bin/cloudflared',
            f'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{cf_arch}')
        def refresh_quick_url(proc_ref, wait_seconds=60):
            deadline_q=time.monotonic()+wait_seconds
            while time.monotonic()<deadline_q:
                if proc_ref.poll() is not None:
                    return None
                match_q=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',ctx['ct_tail']('cloudflared',100))
                if match_q:
                    new_url=match_q.group(0)
                    if new_url!=ctx.get('PUBLIC_URL'):
                        ctx['PUBLIC_URL']=new_url
                        print(f'CT_PUBLIC_URL={new_url}',flush=True)
                    return new_url
                time.sleep(1)
            return None
        ctx['ct_refresh_public_url']=refresh_quick_url
        def start():
            args=[binary,'tunnel','--no-autoupdate','--protocol','http2']
            if ctx['TUNNEL_TOKEN']:
                args+=['run','--token',ctx['TUNNEL_TOKEN']]
            else:
                args+=['--url',f'http://127.0.0.1:{ctx["PUBLIC_PORT"]}']
            proc_new=spawn('cloudflared',args)
            if not ctx['TUNNEL_TOKEN'] and ctx.get('PUBLIC_URL'):
                threading.Thread(target=refresh_quick_url,args=(proc_new,),daemon=True).start()
            return proc_new
        key='cloudflared'
    proc=start()
    ctx['PROCS'][key]=(proc,start)
    ctx['TUNNEL_PROCESS_KEY']=key
    deadline=time.monotonic()+180
    registered=False
    while time.monotonic()<deadline:
        if proc.poll() is not None:
            raise RuntimeError(f'{key} exited before registration; see /content/ct/{key}.log')
        log=ctx['ct_tail'](key,100)
        if transport=='frp':
            registered=registered or ('start proxy success' in log and config['proxy_name'] in log)
            if registered:
                url='https://'+ctx['PUBLIC_HOST']
                try:
                    with urllib.request.urlopen(url,timeout=5) as response:
                        if response.status<600:
                            break
                except urllib.error.HTTPError as error:
                    # A valid public TLS response proves the transport/site.
                    # 502 is expected while a large model is still loading;
                    # the service's keep-alive cell gates CT_READY on health.
                    code=error.code
                    error.close()
                    if code<600:
                        break
                except OSError:
                    pass
        elif ctx['TUNNEL_TOKEN'] and 'Registered tunnel connection' in log:
            url='https://'+ctx['PUBLIC_HOST']
            break
        elif not ctx['TUNNEL_TOKEN']:
            match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',log)
            if match:
                url=match.group(0)
                break
        time.sleep(1)
    else:
        raise RuntimeError(f'{key} not ready; check tunnel logs, DNS, HTTPS and server ports')
    ctx['PUBLIC_URL']=url
    print(f'CT_PUBLIC_URL={url}',flush=True)

run_tunnel(globals())


In [ ]:
# @title 5. Ready + keep-alive -- auto-stepdown on OOM, verify auth/context/vision, smoke completion & supervise
BROWSER_UA = "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/132.0.0.0 Safari/537.36"

def http_json(url, key=None, payload=None, timeout=15):
    headers = {"Content-Type": "application/json", "User-Agent": BROWSER_UA}
    if key is not None:
        headers["Authorization"] = "Bearer " + key
    req = urllib.request.Request(
        url,
        headers=headers,
        data=None if payload is None else json.dumps(payload).encode("utf-8"),
    )
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            return resp.status, json.loads(resp.read().decode("utf-8"))
    except urllib.error.HTTPError as err:
        return err.code, {}

def auth_check(base, key):
    for candidate, expected in ((None, 401), ("invalid-deployment-test-key", 401), (key, 200)):
        code, _ = http_json(base + "/v1/models", candidate)
        if code != expected:
            raise RuntimeError(f"Authentication test failed on {base}: expected {expected}, got {code}")

LOCAL_BASE = f"http://127.0.0.1:{SERVICE_PORT}"
ladder = list(HW_PLAN.get("context_ladder") or [HW_PLAN["context"]])
ub_ladder = list(HW_PLAN.get("ubatch_ladder") or [HW_PLAN["ubatch"]])
rung_idx = 0
ub_idx = 0
deadline = time.monotonic() + 2400
last_notice = 0.0

while time.monotonic() < deadline:
    proc = PROCS["service"][0]
    if proc.poll() is not None:
        recent_log = ct_log_since("llama-server", SPAWN_LOG_OFFSET)
        if looks_like_oom(recent_log) and ub_idx + 1 < len(ub_ladder):
            # Prefer a smaller micro-batch (slower prefill) over a smaller context window
            ub_idx += 1
            prev_ub = HW_PLAN["ubatch"]
            HW_PLAN["ubatch"] = ub_ladder[ub_idx]
            dslog(f"Startup OOM at ubatch={prev_ub}; stepping down to ubatch={HW_PLAN['ubatch']} and retrying")
            PROCS["service"] = (start_service(), start_service)
            continue
        if HW_PLAN.get("context_auto") and rung_idx + 1 < len(ladder) and looks_like_oom(recent_log):
            rung_idx += 1
            prev_ctx = HW_PLAN["context"]
            HW_PLAN["context"] = ladder[rung_idx]
            dslog(f"Startup OOM at ctx={prev_ctx}; stepping down to ctx={HW_PLAN['context']} and retrying")
            PROCS["service"] = (start_service(), start_service)
            continue
        raise RuntimeError("llama-server exited during load:\n" + ct_tail("llama-server", 60))
    try:
        if http_json(LOCAL_BASE + "/health", API_TOKEN, timeout=10)[0] == 200:
            break
    except (OSError, ValueError):
        pass
    if time.monotonic() - last_notice > 45:
        dslog(f"Waiting for model load / warmup (ctx={HW_PLAN['context']}) | GPU: {ct_gpu_stat()}")
        last_notice = time.monotonic()
    time.sleep(5)
else:
    raise RuntimeError("Model load timeout:\n" + ct_tail("llama-server", 60))

if should_evict_page_cache(HW_PLAN):
    evict_gguf_page_cache(MODEL_PATH.parent, keep_engram=not HW_PLAN.get("engram_in_ram", False), log=dslog)

auth_check(LOCAL_BASE, API_TOKEN)
code, props = http_json(LOCAL_BASE + "/props", API_TOKEN, timeout=20)
allocated = (props.get("default_generation_settings") or {}).get("n_ctx")
if code != 200 or allocated != HW_PLAN["context"]:
    raise RuntimeError(f"Allocated context ({allocated}) differs from target ({HW_PLAN['context']})")
vision_enabled = bool((props.get("modalities") or {}).get("vision", False))
if PROJECTOR_PATH is not None and not vision_enabled:
    raise RuntimeError("Vision projector configured but server does not advertise vision")
dslog(
    f"Local verified: auth=401/401/200, n_ctx={allocated}, vision={vision_enabled}, "
    f"mode={HW_PLAN['mode']}, engram_in_ram={HW_PLAN['engram_in_ram']}"
)

ct_public_gate(
    PUBLIC_URL,
    lambda: http_json(PUBLIC_URL + "/v1/models", API_TOKEN, timeout=15)[0],
    lambda: auth_check(PUBLIC_URL, API_TOKEN),
    dslog,
    timeout=180,
    interval=8,
)
smoke_payload = {
    "model": DS_CFG["alias"],
    "messages": [{"role": "user", "content": "Reply with the exact token OK"}],
    "max_tokens": 32,
    "temperature": 0,
}
if DS_CFG["reasoning"] != "off":
    smoke_payload["chat_template_kwargs"] = {"enable_thinking": False}
code, resp = http_json(
    PUBLIC_URL + "/v1/chat/completions",
    API_TOKEN,
    smoke_payload,
    timeout=240,
)
if code != 200:
    raise RuntimeError(f"Public smoke completion failed: HTTP {code}")
msg_obj = ((resp.get("choices") or [{}])[0].get("message") or {})
reply = msg_obj.get("content") or msg_obj.get("reasoning_content") or ""
dslog(f"PUBLIC VERIFIED: {PUBLIC_URL}/v1 | n_ctx={allocated} | reply={reply.strip()[:60]!r}")

ver_rec = {
    "service": SERVICE,
    "verified_at": time.strftime("%FT%T%z"),
    "base_url": PUBLIC_URL + "/v1",
    "repo": DS_CFG["repo"],
    "alias": DS_CFG["alias"],
    "mode": HW_PLAN["mode"],
    "n_cpu_moe": HW_PLAN["n_cpu_moe"],
    "engram_in_ram": HW_PLAN["engram_in_ram"],
    "default_reasoning": DS_CFG["reasoning"],
    "context_allocated": allocated,
    "vision_enabled": vision_enabled,
    "reply": reply[:120],
}
write_json_atomic(DS_SERVICE_DIR / "last-verification.json", ver_rec)

print(f"{SERVICE} is up: {PUBLIC_URL} (n_ctx={allocated}, mode={HW_PLAN['mode']})")
print("CT_READY", flush=True)

def ds_heartbeat(uptime):
    gpu_mem = ct_gpu_stat("memory.used,memory.total,utilization.gpu")
    try:
        write_json_atomic(
            DS_SERVICE_DIR / "last-status.json",
            {
                "service": SERVICE,
                "kernel_pid": os.getpid(),
                "server_pid": PROCS["service"][0].pid,
                "tunnel_pid": PROCS[TUNNEL_PROCESS_KEY][0].pid,
                "context": HW_PLAN["context"],
                "mode": HW_PLAN["mode"],
                "engram_in_ram": HW_PLAN["engram_in_ram"],
                "alias": DS_CFG["alias"],
                "url": PUBLIC_URL + "/v1",
                "updated_at": time.strftime("%FT%T%z"),
                "uptime_sec": uptime,
                "gpu_memory": gpu_mem,
            },
        )
    except OSError:
        pass
    dslog(f"heartbeat | up={uptime}s | n_ctx={HW_PLAN['context']} | GPU: {gpu_mem}")

ct_watchdog(
    lambda: http_json(LOCAL_BASE + "/health", API_TOKEN, timeout=10)[0] == 200,
    start_service,
    dslog,
    log_name="llama-server",
    heartbeat=ds_heartbeat,
)
